# Assignment 1. Exploratory Modeling with JUSTICE

**Course:** EPA141A Model Based Decision Making — Delft University of Technology  
**Model:** JUSTICE
**Contact:** J.ZatarainSalazar@tudelft.nl  

---

## Learning Outcomes

After completing this assignment you will be able to:
1. Verify that the JUSTICE model is installed and runs correctly.
2. Identify and define **model input uncertainties**.
3. Use **Latin Hypercube Sampling** to generate a scenario ensemble that spans the uncertainty space.
4. Run JUSTICE across many scenarios under different policy assumptions and collect the outcomes.
5. Visualise outcome **distributions**, **pair plots**, and **time series envelopes** to characterise how uncertainty propagates.


---

## Background

When building models of complex systems (climate, economy, ecology) we rarely know
the exact value of every parameter. A damage-function coefficient, a discount rate,
or a climate sensitivity parameter are all uncertain. **Exploratory modeling** treats
uncertainty explicitly: we run *many* scenarios, sampling the uncertain parameters
systematically, and study the resulting spread in model outputs.

In this assignment we treat four parameters as uncertain:

| Symbol | Parameter | Default | Range | Description |
|--------|-----------|---------|-------|-------------|
| `ρ` | Pure rate of social time preference | 0.015 | [0.001, 0.030] | Contested in the Stern–Nordhaus debate |
| `η` | Elasticity of marginal utility | 1.45 | [0.5, 1.5] | Empirical estimates range widely |
| `δ` | Damage function scale factor | 1.0 | [0.5, 2.0] | Structural uncertainty in damage estimates |
| `ecs_ensemble` | FaIR ensemble member index | 1 | [1, 1001] | Selects a calibrated ECS parameter set |

`ecs_ensemble` is a *physical* uncertainty (how sensitive the climate system is to CO₂),
while `ρ`, `η`, and `δ` are *normative* uncertainties (contested values in economic ethics).

We also compare two **policies** that differ in their emission control rate:

| Policy | `ecr_plateau` | Description |
|--------|--------------|-------------|
| `no_abatement` | 0.0 | Zero mitigation effort throughout the simulation |
| `moderate_abatement` | 0.4 | 40 % emission reduction applied from the start |

---

## Step 0 — Verify your installation

Run the cell below. Every line should end with ✓.  
If any line shows ✗, follow the installation instructions in setup_and_orientation.html

In [6]:
import importlib, sys
required = ["justice", "numpy", "pandas", "matplotlib",
            "ema_workbench", "scipy", "seaborn"]
for pkg in required:
    found = importlib.util.find_spec(pkg) is not None
    print(f"  {'✓' if found else '✗'}  {pkg}")
print(f"\nPython {sys.version.split()[0]}")

  ✓  justice
  ✓  numpy
  ✓  pandas
  ✓  matplotlib
  ✓  ema_workbench
  ✓  scipy
  ✓  seaborn

Python 3.14.3


## Setup — Imports and model configuration

The cell below imports all required packages and applies a Python 3.14 compatibility patch for `matplotlib.path.Path.__deepcopy__`. It also configures EMA Workbench logging and defines the shared name lists for outcomes, parameters, and policies that are referenced throughout the notebook.

## Step 1 — Run JUSTICE with default parameters

Before exploring uncertainty, confirm the model runs with its default settings. Run the cell before anything else, it loads all required packages and defines shared name lists used throughout the notebook.


**Task 1.1** —  the `justice_model` function below does the following:
1. Hard-reset JUSTICE and instantiate a fresh model with the given `ecs_ensemble` index.
2. Set `rho` on `model.welfare_function.pure_rate_of_social_time_preference`.
3. Set `eta` on `model.welfare_function.elasticity_of_marginal_utility_of_consumption`.
4. Scale the three active damage coefficients (`coefficient_a`, `coefficient_b`, `damage_gdp_ratio_with_gradient`) by `delta`.
5. Run with zero abatement (`ECR = 0`) and return all four outcomes as a dict.

> **Note:** Default argument values are required by the EMA Workbench function model interface.

**Task 1.2** — Run the default case (`ecs_ensemble=1`, all other parameters at their defaults) and record the four outcome values.

In [1]:
import os, sys
# ── Add JUSTICE-main to sys.path so justice internal imports resolve ───────────
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
_justice_root = os.path.normpath(os.path.join(_NOTEBOOK_DIR, '../JUSTICE-main'))

_PLOTS_DIR = os.path.join(_NOTEBOOK_DIR, "plots")
os.makedirs(_PLOTS_DIR, exist_ok=True)
if _justice_root not in sys.path:
    sys.path.insert(0, _justice_root)
os.chdir(_justice_root)

import warnings; warnings.filterwarnings("ignore")
import logging
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from ema_workbench import (
    Model, RealParameter, ScalarOutcome, ArrayOutcome,
    perform_experiments, ema_logging, Sample,
    SequentialEvaluator, MultiprocessingEvaluator,
)
from justice.model import JUSTICE
from justice.util.enumerations import WelfareFunction
from justice.objectives.objective_functions import years_above_temperature_threshold

ema_logging.log_to_stderr(logging.WARNING)   # suppress per-run INFO noise

OBJECTIVES   = ["welfare", "years_above_temperature_threshold",
                "welfare_loss_damage", "welfare_loss_abatement"]
PARAMS       = ["rho", "eta", "delta", "ecs_ensemble"]
POLICY_NAMES = ["no_abatement", "moderate_abatement"]

print("Imports OK")
import matplotlib.path as _mpath
def _patched_path_deepcopy(self, memo=None):
    if memo is None: memo = {}
    new_path = _mpath.Path.__new__(_mpath.Path)
    memo[id(self)] = new_path
    verts = self._vertices.copy()
    codes = self._codes.copy() if self._codes is not None else None
    new_path.__init__(verts, codes,
                      _interpolation_steps=self._interpolation_steps, readonly=False)
    return new_path
_mpath.Path.__deepcopy__ = _patched_path_deepcopy

Imports OK


In [2]:
def justice_model(rho=0.015, eta=1.45, delta=1.0, ecs_ensemble=1, ecr_plateau=0.0):
    """EMA Workbench function model — configurable abatement, Utilitarian welfare.

    Parameters
    ----------
    ecr_plateau : float
        Emission control rate applied uniformly across all regions and timesteps.
        0.0 = no abatement; 0.4 = 40% abatement (moderate policy).
    """
    JUSTICE.hard_reset()
    ensemble_idx = int(np.round(np.clip(ecs_ensemble, 1, 1001)))
    model = JUSTICE(
        start_year=2015, end_year=2300, timestep=1,
        scenario=2, climate_ensembles=ensemble_idx, stochastic_run=False,
        social_welfare_function=WelfareFunction.UTILITARIAN,
    )
    model.economy.pure_rate_of_social_time_preference = float(rho)
    model.economy.elasticity_of_marginal_utility_of_consumption = float(eta)
    model.welfare_function.pure_rate_of_social_time_preference = float(rho)
    model.welfare_function.elasticity_of_marginal_utility_of_consumption = float(eta)
    model.damage_function.coefficient_a                  *= float(delta)
    model.damage_function.coefficient_b                  *= float(delta)
    model.damage_function.damage_gdp_ratio_with_gradient *= float(delta)

    ecr = np.full(model.emission_control_rate.shape[:2], float(ecr_plateau))
    model.run(emission_control_rate=ecr, endogenous_savings_rate=True)
    datasets = model.evaluate()

    welfare = float(np.abs(np.squeeze(datasets["welfare"])))
    yat     = float(np.squeeze(
        years_above_temperature_threshold(datasets["global_temperature"], 2.0)
    ))
    _, _, _, wl_dmg = model.welfare_function.calculate_welfare(
        datasets["damage_cost_per_capita"], welfare_loss=True)
    _, _, _, wl_abt = model.welfare_function.calculate_welfare(
        datasets["abatement_cost_per_capita"], welfare_loss=True)

    # Global mean temperature trajectory (shape: n_timesteps)
    temp = np.squeeze(datasets["global_temperature"])
    if temp.ndim == 2:
        temp = temp.mean(axis=0)
    temperature_trajectory = temp.astype(float)

    return {
        "welfare":                          welfare,
        "years_above_temperature_threshold": yat,
        "welfare_loss_damage":              float(np.abs(np.squeeze(wl_dmg))),
        "welfare_loss_abatement":           float(np.abs(np.squeeze(wl_abt))),
        "temperature_trajectory":           temperature_trajectory,
    }

# Smoke test
test = justice_model()
for k, v in test.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: array shape {v.shape}, range [{v.min():.2f}, {v.max():.2f}]")
    else:
        print(f"  {k}: {v:.4f}")


  welfare: 103.7211
  years_above_temperature_threshold: 259.0000
  welfare_loss_damage: 3980.5410
  welfare_loss_abatement: 74364.1321
  temperature_trajectory: array shape (286,), range [1.20, 6.01]


## Step 2 — Define uncertain parameters

**Task 2.1** — Complete the cell below by:
1. Creating a `Model` object wrapping `justice_model`.
2. Assigning `uncertainties` — four `RealParameter` objects matching the table in the Background.
3. Assigning `outcomes` — four `ScalarOutcome` objects.

**Task 2.2** — Explain in 2–3 sentences why a *higher* pure rate of time preference `ρ` tends to lead to *less* stringent near-term mitigation. Reference the Ramsey rule in your answer.

The Ramsey rule sets the social discount rate as $r = \rho + \eta \cdot g$, where $g$ is per-capita consumption growth. A higher $\rho$ raises the discount rate, which reduces the present value of future climate damages. Because the costs of mitigation are paid today while most benefits accrue in the distant future, a higher discount rate makes abatement look less worthwhile in cost-benefit terms, leading the model to prescribe less stringent near-term emission reductions.

**Task 2.3** — Explain why `ecs_ensemble` is a *physical* uncertainty while `ρ`, `η`, `δ` are *normative* uncertainties. What does this distinction imply about which outcomes each type of uncertainty can influence?

`ecs_ensemble` captures genuine scientific uncertainty about the climate system's equilibrium climate sensitivity — in principle, better observations could narrow this range. By contrast, `ρ`, `η`, and `δ` reflect contested ethical and economic value judgements (how much we discount the future, how we weigh inequality, how large we believe damage costs are), and no amount of empirical data can definitively resolve them. Physical uncertainty primarily propagates into temperature-related outcomes (e.g., years above 2 °C), whereas normative uncertainties drive welfare-based outcomes (welfare, welfare loss from damage and abatement) — though in practice both types interact across all outcomes because the economic and climate sub-models are coupled.

In [ ]:
em_model = Model('JUSTICE', function=justice_model)

em_model.uncertainties = [
    RealParameter("rho",          0.001, 0.030),
    RealParameter("eta",          0.5,   1.5),
    RealParameter("delta",        0.5,   2.0),
    RealParameter("ecs_ensemble", 1,     1001),
]

# Policy lever: emission control rate applied uniformly across all timesteps
em_model.levers = [
    RealParameter("ecr_plateau", 0.0, 1.0),
]

em_model.outcomes = [
    ScalarOutcome("welfare"),
    ScalarOutcome("years_above_temperature_threshold"),
    ScalarOutcome("welfare_loss_damage"),
    ScalarOutcome("welfare_loss_abatement"),
    ArrayOutcome("temperature_trajectory"),
]

print(f"Uncertainties : {[u.name for u in em_model.uncertainties]}")
print(f"Levers        : {[l.name for l in em_model.levers]}")
print(f"Outcomes      : {[o.name for o in em_model.outcomes]}")

## Step 3 — Latin Hypercube Sampling ensemble

**Latin Hypercube Sampling (LHS)** divides each uncertain dimension into equally
probable intervals and samples exactly once from each interval. Compared to Monte Carlo,
it guarantees better coverage of the full parameter space with fewer samples.

You first need to define two policies to compare: one with no abatement (ecr_plateau=0.0) and one with moderate abatement (ecr_plateau=0.4). Use Sample to give each policy a name.

`perform_experiments` uses LHS by default. A single call handles sampling, execution, and result collection.

**Task 3.1** — Run 100 scenarios using `SequentialEvaluator`. Store the results as:
- `experiments` — a DataFrame with one row per scenario and one column per parameter
- `outcomes` — a dict mapping outcome names to arrays

In [ ]:
policies = [
    Sample("no_abatement",       ecr_plateau=0.0),
    Sample("moderate_abatement", ecr_plateau=0.4),
]
  

# tqdm compatibility fix for Python ≥ 3.14 (not needed on the course environment)
import tqdm as _tqdm_mod
if _tqdm_mod.tqdm is not _tqdm_mod.std.tqdm:
    _tqdm_mod.tqdm = _tqdm_mod.std.tqdm

# Note: MultiprocessingEvaluator gives large speed-ups for 500+ scenarios when
# run from a script (python run_experiments.py). Inside a Jupyter notebook the
# spawned worker processes cannot reliably re-import the venv, so we use
# SequentialEvaluator here. For script-based runs, swap the context manager.
with SequentialEvaluator(em_model) as evaluator:
    experiments, outcomes = evaluator.perform_experiments(100, policies=policies)

df_results = pd.DataFrame({k: v for k, v in outcomes.items() if k != 'temperature_trajectory'})
df_params  = experiments[PARAMS]
df_results['policy'] = experiments['policy'].values

print(f"Ensemble complete: {len(experiments)} runs  "
      f"({df_results['policy'].value_counts().to_dict()})")
print(df_results[OBJECTIVES].agg(['min', 'median', 'max']).round(2).to_string())

## Step 4 — Visualise outcome distributions

**Task 4.1** — Run the cell below. It plots the distribution of each outcome across your 100 LHS scenarios, separately for each policy. The red dashed line is the model's default output (all parameters at their reference values).

Look at the four plots and answer:

**Which outcomes show a wide spread? What does that tell you about uncertainty?**
All four outcomes display a wide spread, but `welfare_loss_abatement` and `welfare` show the largest relative range (multiple orders of magnitude on the log scale). This confirms that uncertainty in `ρ`, `η`, `δ`, and the climate ensemble propagates substantially through the model — a single default run is far from representative of the full range of plausible futures.

**Does the distribution shift between the two policies? For which outcomes?**
The two policies separate most clearly for `years_above_temperature_threshold` (moderate abatement shifts the distribution toward fewer years above 2 °C) and `welfare_loss_abatement` (the moderate-abatement distribution shifts to higher values because abatement costs are now non-zero). `welfare_loss_damage` shifts in the opposite direction under moderate abatement (lower damages). `welfare` shows more overlap between policies, reflecting the trade-off between reduced damages and increased abatement costs.

**Where does the default (red line) sit relative to the distributions — is it typical or an outlier?**
For most outcomes the default sits near the centre of the no-abatement distribution, suggesting its parameter values are close to central tendency. However, for `welfare_loss_abatement` it sits at or near zero (the no-abatement baseline), making it an outlier relative to the moderate-abatement distribution. This illustrates why relying on the default alone can be misleading when comparing policy options.

In [ ]:
defaults = justice_model()

palette = {'no_abatement': 'steelblue', 'moderate_abatement': 'darkorange'}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, obj in zip(axes.flat, OBJECTIVES):
    for pol, grp in df_results.groupby('policy'):
        data = grp[obj]
        use_log = (data.max() / (data.min() + 1e-12)) > 100
        vals = np.log10(data + 1e-12) if use_log else data
        ax.hist(vals, bins=25, color=palette[pol], edgecolor='white',
                alpha=0.55, label=pol.replace('_', ' '))
    ax.axvline(
        np.log10(defaults[obj] + 1e-12) if use_log else defaults[obj],
        color='crimson', lw=2, ls='--', label='Default (no abatement)',
    )
    ax.set_xlabel('log₁₀(Value)' if use_log else 'Value')
    ax.set_title(obj.replace('_', ' '), fontsize=10)
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)

fig.suptitle("Outcome distributions — 100 LHS scenarios × 2 policies", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(_PLOTS_DIR, "a01ema_outcome_distributions.png"), dpi=150, bbox_inches='tight')
plt.show()


## Step 5 — Outcome pair plots

Run the cell below. It produces a pair plot: every outcome plotted against every other outcome, coloured by policy.

Look at the plot and answer:

**Which pairs of outcomes move together (are correlated)? Which don't?**
`welfare` and `welfare_loss_damage` tend to move together — scenarios with high damage losses also show lower net welfare. `years_above_temperature_threshold` correlates positively with `welfare_loss_damage`: hotter scenarios incur more damage. By contrast, `welfare_loss_abatement` is largely uncorrelated with temperature outcomes, because abatement costs are driven mainly by `ρ` and `η` rather than by the climate ensemble.

**Do the two policies separate clearly in any of the panels, or do they overlap?**
The policies separate most clearly in panels involving `years_above_temperature_threshold` (moderate abatement produces consistently fewer years above 2 °C) and `welfare_loss_abatement` (moderate abatement has non-zero abatement costs, pushing its cluster upward). In the `welfare` panels the two policy clouds overlap substantially, indicating that uncertainty in normative parameters can produce similar welfare levels under either policy depending on the scenario.

**What does it mean for decision-making if two outcomes are strongly correlated?**
Strong correlation means that improving on one outcome automatically improves the other — the decision maker does not face a trade-off between them. This simplifies the problem: policies can be compared on fewer independent dimensions. Conversely, uncorrelated or negatively correlated outcomes reveal genuine trade-offs that require explicit value judgements about what to prioritise.

In [ ]:
df_outcomes_plot = df_results[OBJECTIVES + ['policy']].copy()

for col in ['welfare_loss_damage', 'welfare_loss_abatement', 'welfare']:
    df_outcomes_plot[col] = np.log10(df_outcomes_plot[col] + 1e-9)

log_labels = {
    'welfare': 'log₁₀ welfare',
    'welfare_loss_damage': 'log₁₀ wl_damage',
    'welfare_loss_abatement': 'log₁₀ wl_abatement',
    'years_above_temperature_threshold': 'years above 2 °C',
}
df_outcomes_plot.rename(columns=log_labels, inplace=True)
plot_vars = list(log_labels.values())

g = sns.pairplot(
    df_outcomes_plot, vars=plot_vars,
    hue='policy', palette=palette,
    plot_kws={'alpha': 0.35, 's': 12}, diag_kind='kde',
)
g.fig.suptitle("Outcome pair plot — 100 scenarios × 2 policies", y=1.02, fontsize=11)
plt.savefig(os.path.join(_PLOTS_DIR, "a01ema_outcome_pairplot.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: a01ema_outcome_pairplot.png")


## Step 6 — Temperature trajectory envelope

Run the cell below. It plots the temperature trajectory for all 100 scenarios as an ensemble envelope: the median line, the interquartile range (dark band), and the 5–95th percentile range (light band).

Look at the plot and answer:

**At year 2100, does the moderate abatement policy reliably stay below 2 °C, or does uncertainty still push some trajectories above the threshold?**
Even under moderate abatement (40 % emission reduction), the 5–95 % envelope at 2100 typically straddles the 2 °C line — the median may be close to or just above the threshold, but high-ECS scenarios in the upper tail clearly exceed it. The 2 °C target is therefore not reliably achieved by this policy level alone.

**How wide is the envelope at 2100 compared to 2050? What drives that widening?**
The envelope is noticeably wider at 2100 than at 2050. This widening reflects the compounding effect of physical uncertainty: different climate sensitivity values (`ecs_ensemble`) lead to small temperature differences early on, but these diverge further over decades as cumulative CO₂ concentrations differ across scenarios. Longer time horizons give structural uncertainty more time to separate trajectories.

**If you were advising a policymaker, what would you tell them about the 2 °C target based on this plot?**
40 % emission reduction is insufficient to reliably keep warming below 2 °C across the plausible range of climate sensitivities. Policymakers should not treat the median trajectory as the likely outcome — the upper tail of the envelope represents low-probability but high-consequence futures that responsible planning must account for. Stronger mitigation (higher `ecr_plateau`) would be needed to narrow the envelope and push the median and upper tail below the threshold with greater confidence.

In [ ]:
temp_all = outcomes['temperature_trajectory']
policy_col = experiments['policy'].values
n_timesteps = temp_all.shape[1]
years = np.arange(2015, 2015 + n_timesteps)

fig, ax = plt.subplots(figsize=(10, 5))

for pol, colour in palette.items():
    mask = policy_col == pol
    traj = temp_all[mask]
    med  = np.median(traj, axis=0)
    p25  = np.percentile(traj, 25, axis=0)
    p75  = np.percentile(traj, 75, axis=0)
    p05  = np.percentile(traj,  5, axis=0)
    p95  = np.percentile(traj, 95, axis=0)
    ax.fill_between(years, p05, p95, alpha=0.15, color=colour)
    ax.fill_between(years, p25, p75, alpha=0.35, color=colour)
    ax.plot(years, med, color=colour, lw=2, label=pol.replace('_', ' ') + ' (median)')

ax.axhline(2.0, color='black', lw=1.2, ls='--', label='2 °C threshold')
ax.set_xlabel('Year')
ax.set_ylabel('Global mean temperature anomaly (°C above pre-industrial)')
ax.set_title('Temperature trajectory ensemble — 100 scenarios × 2 policies\n'
             'Shaded bands: 5–95 % (light) and IQR (dark)')
ax.legend(fontsize=9)
ax.set_xlim(years[0], years[-1])

# run the code below if you want to save the images in the plot directory
# plt.tight_layout()
# plt.savefig(os.path.join(_PLOTS_DIR, "a01ema_temperature_envelope.png"), dpi=150, bbox_inches='tight')
# plt.show()
# print("Saved: a01ema_temperature_envelope.png")


## Reflection Questions

**1. Coverage vs. efficiency.** Why does LHS typically need fewer samples than simple random (Monte Carlo) sampling to achieve the same coverage of the parameter space?

LHS stratifies each uncertain dimension into $n$ equally probable intervals and draws exactly one sample from each interval. This enforces that every region of the marginal distribution is represented, eliminating large gaps that random sampling can produce by chance. As a result, LHS achieves the same variance reduction in output statistics with far fewer model runs — particularly important when each run is computationally expensive, as is the case with integrated assessment models like JUSTICE.

**2. Default as reference.** In your histogram plots, does the default run fall near the centre of the distribution or near one of the tails? What does this imply about how representative a single default run is?

The default run (red dashed line) tends to fall near the centre of the distributions for most outcomes, consistent with its parameter values being mid-range defaults. However, even when it lands near the median, the wide spread of the distributions shows that the default outcome is just one of many plausible outcomes. Relying on a single default run would give an overconfident and potentially misleading picture of system behaviour — it cannot represent the tails where the most consequential (best or worst) outcomes cluster.

**3. Propagation of uncertainty.** Which of the four outcomes shows the widest relative spread? Propose a hypothesis explaining why that outcome is more sensitive to the chosen uncertain parameters.

`welfare_loss_abatement` typically shows the widest relative spread (spanning several orders of magnitude on the log scale). A likely explanation is that abatement cost is highly sensitive to the elasticity of marginal utility `η` and the time preference `ρ` simultaneously: these parameters jointly determine how future streams of abatement costs are aggregated into a welfare-equivalent lump sum. Small changes in either parameter compound over the long simulation horizon (2015–2300), amplifying differences in the integrated welfare loss. The `delta` parameter further multiplies this effect through its scaling of damage costs, which interact with abatement decisions via the welfare function.